# QF 627 Programming and Computational Finance
## Lesson 07 | Unsupervised Learning, PCA, and Portfolio Management

> Hi, Team 👋 Thank you for opening the lecture note 🙂

![pca](https://images.squarespace-cdn.com/content/v1/53f3eb3ce4b077de0318f4ea/1628755215426-O3WRR9TNVXU0UFQ2HB68/PCA.gif?format=2500w "pca")

> Today, we will continue our learning in machine learning. We will start off by exploring how to use clustering for Singapore condominium rental prices. Then, we will move to another use case of machine learning in the quantitative (process-driven) investing, specifically portfolio management and asset allocation.

## DEPENDENCIES

In [ ]:
# # Load libraries.

# import numpy as np
# import pandas as pd

# import matplotlib.pyplot as plt
# import matplotlib as mpl

# import seaborn as sns

# import time

# import datetime
# import re

# # for commenting purposes

# # import altair as alt
# # import plotly.express as px
# # import xlrd

# import statsmodels.api as sm

# import warnings
# warnings.filterwarnings("ignore")

# import pandas_datareader.data as web
# from pandas_datareader import data as pdr

# import yfinance as yf

# # yf.pdr_override()

# # Setting baseline seed
# np.random.seed(2025)

# # Set print options.

# np.set_printoptions(precision = 3)

# plt.style.use("ggplot")

# mpl.rcParams["axes.grid"] = True
# mpl.rcParams["grid.color"] = "grey"
# mpl.rcParams["grid.alpha"] = 0.25

# mpl.rcParams["axes.facecolor"] = "white"

# mpl.rcParams["legend.fontsize"] = 14

# %matplotlib inline

# # Define our customized timer function

# def countdown(Time):
    
#     while Time:
#         minutes, seconds = divmod(Time, 60)
#         timer = "{:02d}:{:02d}".format(minutes, seconds)
        
#         print(timer, end = "\r")
#         time.sleep(1)
#         Time -= 1
        
#     print("Let us solve the problem above together :)")

In [ ]:
%whos

## 👉 <a id = "top">Learning Pointers</a> 👈 

## [1. Clustering for Mapping Singapore Condominium Rental Prices](#p1)

> ### <font color = red> Clustering for Analytics in Real Estate </font>

## [2. PCA for Portfolio Management and Asset Allocation](#p2)

> ### <font color = red> Yet Another Use Case of Unsupervised Machine Learning in Finance </font>

## <a id = "p1">1.</a>  <font color = "green"> Clustering for Mapping Singapore Condominium Rental Prices </font>  [back to table of contents](#top)

### Defining Analytic Question: Your Quant & Machine Learning Questions 😊

> You are a data scientist for a real estate firm in Singapore, and you have been tasked with analyzing the rental market for condominiums. The goal is to identify distinct rental price segments to help the firm understand market trends, optimize rental pricing strategies, and provide insights for property investors. You have a dataset containing various attributes of condominium rentals.

#### Dataset Description

> The dataset includes the following variables for each condominium rental:

* `unit_id`: A unique identifier for each condominium unit.
* `property`: The name of the property.
* `district`: The district where the condominium is located.
* `loc_x`: The x-coordinate of the condominium location.
* `loc_y`: The y-coordinate of the condominium location.
* `region`: The region where the condominium is located (e.g., CCR, RCR, OCR).
* `ref_year`: The reference year of the rental data.
* `age`: The age of the condominium in years.
* `dist_to_mrt`: The distance of the condominium to the nearest MRT station in kilometers.
* `price_median`: The median rental price of the condominium in SGD per square meter.

#### Objective

> The objective is to segment the condominiums into distinct rental price categories based on their rental price, location, age, and proximity to MRT stations. By identifying these segments, the firm can develop targeted rental pricing strategies, provide better recommendations to clients, and identify investment opportunities.

### Dependencies

In [ ]:
# # Install necessary packages if not already installed

# !pip install plotly geopandas folium

In [ ]:
# import plotly.express as px
# import folium
# from folium.plugins import MarkerCluster

### IMPORT

In [ ]:
# # Load the dataset
# price =\
# (
#     pd
#     .read_csv("https://talktoroh.com/s/rental_price.csv")
# )

In [ ]:
# # Display the summary of the dataset

# price \
#     .info()

### WRANGLE

In [ ]:
# # Subsetting the data

# price_subset =\
#     price[["price_median", "loc_x", "loc_y"]]

In [ ]:
# # Run summary statistics

# price_subset \
#     .describe()

In [ ]:
# from sklearn.preprocessing import StandardScaler

In [ ]:
# # Normalize the numerical variables

# scaler = StandardScaler()

# price_subset_normalized =\
# (
#     scaler
#     .fit_transform(price_subset)
# )

In [ ]:
# # Convert the normalized data back to a DataFrame

# price_subset_normalized_df =\
# (
#     pd
#     .DataFrame(price_subset_normalized, 
#                columns = price_subset.columns)
# )

In [ ]:
# # Run the summary statistics of the normalized data

# price_subset_normalized_df \
#     .describe()

### MODEL

In [ ]:
# from sklearn.cluster import KMeans

### Elbow Method: Calculating `WSS` (Within-Cluster Sum of Squares)

In [ ]:
# wss = []

# for k in range(2, 11):
#     kmeans = KMeans(n_clusters=k, random_state=0).fit(price_subset_normalized)
#     wss.append(kmeans.inertia_)

In [ ]:
# plt.figure(figsize=(10, 7)
#           )

# plt.plot(range(2, 11),
#          wss, "go-")

# plt.xlabel("Number of clusters")
# plt.ylabel("With-in Cluster Sum of Squares")

# plt.title("Elbow Method For Optimal Number of Clusters")

### Beyond Within-Cluster Sum of Squares (WSS): `Silhouette` Score

The silhouette score is a measure of how similar an object is to its own cluster (cohesion) compared to other clusters (separation). The silhouette score for a single sample \( i \) is given by:

$
s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}
$

where:
- $ a(i) $ is the average distance between the sample $ i $ and all other points in the same cluster.
- $ b(i) $ is the average distance between the sample $ i $ and all points in the nearest cluster to which $ i $ does not belong.

The silhouette score for the entire dataset is the mean silhouette score for all samples:

$
S = \frac{1}{n} \sum_{i=1}^{n} s(i)
$

where $ n $ is the total number of samples.

### Elaboration

- **Cohesion ($ a(i) $)**: This measures how close each point in a cluster is to other points in the same cluster. It is the average intra-cluster distance. A lower value of $ a(i) $ indicates that the sample is well matched to its own cluster.

- **Separation ($ b(i) $)**: This measures how far the point is from points in the nearest cluster that the point is not a part of. It is the average nearest-cluster distance. A higher value of $ b(i) $ indicates that the sample is poorly matched to its neighboring cluster.

- **Silhouette score $ s(i) $**: The value of $ s(i) $ ranges between -1 and +1.
  - A value close to +1 indicates that the sample is far away from the neighboring clusters, implying that the sample is well clustered.
  - A value close to 0 indicates that the sample is on or very close to the decision boundary between two neighboring clusters.
  - A value close to -1 indicates that the sample might have been assigned to the wrong cluster, as it is closer to a neighboring cluster than to its own cluster.

> The silhouette score is a useful metric to evaluate the quality of clustering. Higher silhouette scores indicate better-defined clusters, whereas lower scores indicate that the clustering may have mixed or overlapping clusters.


In [ ]:
# from sklearn.metrics import silhouette_score

In [ ]:
# # Silhouette Method

# silhouette_scores = []

# for k in range(2, 11):
#     kmeans = (KMeans(n_clusters = k, 
#                     random_state = 627)
#               .fit(price_subset_normalized)
#              )
    
#     score = silhouette_score(price_subset_normalized, kmeans.labels_)

#     silhouette_scores.append(score)

# plt.figure(figsize=(10, 7)
#           )

# plt.plot(range(2, 11), 
#          silhouette_scores, "go-")

# plt.xlabel("Number of clusters")
# plt.ylabel("Silhouette Score")

# plt.title("Silhouette Method For Optimal Number of Clusters")

In [ ]:
# # Perform final clustering with the chosen number of clusters

# optimal_clusters = 3  # Suppose the optimal number of clusters is 3 based on previous methods

# final_model =\
# (
#     KMeans(n_clusters = optimal_clusters, 
#            random_state = 627)
# )

# final_clusters =\
# (
#     final_model
#     .fit_predict(price_subset_normalized)
# )

In [ ]:
# # Add the cluster labels to the original DataFrame
# price["cluster"] = final_clusters

In [ ]:
# # Profile the clusters
# cluster_profile =\
# (    
#     price
#     .groupby("cluster")
#     .agg(
#         {"price_median": ["mean", "std"],
#          "loc_x": "count"}
#          )
#     .reset_index()
# )

In [ ]:
# cluster_profile.columns = ["Cluster", "Mean_Price", "Std_Price", "Count"]

# cluster_profile

In [ ]:
# # Convert the cluster column to a string for plotting

# price["cluster"] =\
# (
#     price["cluster"]
#     .astype(str)
# )

In [ ]:
# from lets_plot import *
# LetsPlot.setup_html()

In [ ]:
# # Create a plot using lets-plot
# plot =\
# (   
#     ggplot(price) + 
#     geom_point(aes(x = "loc_x", 
#                    y = "loc_y", 
#                    color = "cluster"), 
#                size = 2) + 
#     ggtitle("Clusters of Rental Prices") + 
#     theme(axis_title_x = element_text(size = 14), 
#           axis_title_y = element_text(size = 14)
#          ) + 
#     labs(color = "Cluster") +
#     scale_color_discrete()
# )

# # Display the plota
# plot.show()

### Interactive Mapping

In [ ]:
# %whos

In [ ]:
# # Create a Folium map centered around Singapore

# m =\
# (
#     folium
#     .Map(location = [1.3521, 103.8198], 
#          zoom_start = 12)
# )

In [ ]:
# # Save the map to an HTML file
# m.save("interactive_map.html")

In [ ]:
# # Display the map inline (this might not work in all environments, especially in text-based interfaces)
# m

## <a id = "p2">2. </a> <font color = "green"> Unsupervised Learning for Portfolio Management </font>  [back to table of contents](#top)

    PROBLEM STATEMENT
    
> Our goal is to maximize the risk-adjusted returns of an equity portfolio using PCA on a dataset of stock returns. We will use the Dow Jones Industrial Average (DJIA) index and its respective 30 stocks. The return data is extracted from Yahoo! Finance between January 2000 and January 2019. Ultimately, we will compare the performance of our portfolios against a benchmark and backtest the model to assess the effectiveness of the strategy.

    CONTEXT--In these circumstances, dimensionality reduction is useful.
    
> The primary goal of portfolio management is to allocate capital to different asset classes to maximize risk-adjusted returns. Mean-variance portfolio optimization is the most commonly used technique for asset allocation. This method requires an estimated covariance matrix and expected returns of the assets considered.

> Yet, the fluid nature of financial returns leads to estimation errors in these inputs, especially when the number of samples is much smaller than the number of assets being allocated. These errors pose a threat to the optimization of the resultant portfolios and lead to poor and unstable outcomes.

> We will learn how dimensionality reduction addresses the problem. Using PCA, we can take an n × n covariance matrix of our assets and create a set of n linearly uncorrelated principal portfolios (aka eigen portfolio) made up of our assets and their corresponding variances.

> The principal components of the covariance matrix capture most of the covariation among the assets and are mutually uncorrelated. We can use standardized principal components as the portfolio weights, with the statistical warrants that the returns from these principal portfolios are linearly uncorrelated.

#### Let’s get familiar with a general approach to searching for an eigen portfolio for asset allocation. Starting from understanding concepts of PCA, we will ultimately run backtesting of different principal components.

In [ ]:
# import pandas_datareader.data as web
# from pandas_datareader import data as pdr

# import yfinance as yf

## yf.pdr_override()

#### Activate Necessary Packages

> For unsupervised learning

In [ ]:
# from sklearn.decomposition import PCA
# from sklearn.decomposition import KernelPCA

# from sklearn.decomposition import TruncatedSVD

# from numpy.linalg import inv, eig, svd

# from sklearn.manifold import TSNE

> For EDA and Data Transformation

In [ ]:
# from sklearn.preprocessing import StandardScaler

# from pandas.plotting import scatter_matrix

#### IMPORT Data

In [ ]:
# dow =\
# (
#     pd
#     .read_csv("https://talktoroh.com/s/dow_pca-hemx.csv",
#               index_col = 0)
# )

In [ ]:
# dow.head(3)

#### Exploratory Data Analysis (EDA)

In [ ]:
# corr = dow.corr()

# plt.figure(figsize = [16, 16]
#           )

# plt.title("A Heatmap for Correlation Matrix")

# sns \
#     .heatmap(corr,
#              annot = True,
#              cmap = "viridis")

### Data Transformation

> Stocks were added to the index after the start date (here, January 2000). To ensure proper analysis, we will drop those with more than 30% missing values. Two stocks fit this criteria: Dow Chemicals and Visa.

In [ ]:
# missing_values =\
# (
#     dow
#     .isnull() # True (1) vs. False (0)
#     .mean()
#     .sort_values(ascending = False)
# )

# missing_values.head(10)

In [ ]:
# drop_list =\
# (
#     sorted(list(missing_values[missing_values > 0.30]
#                 .index)
#           )
# )

# dow =\
# (
#     dow
#     .drop(labels = drop_list,
#           axis = 1)
# )

# dow.shape[1] == 30 - 2

#### Quick question: Please fill the missing values with the last value available in your data.

In [ ]:
# dow =\
# (
#     dow
#     .fillna(method = "ffill")
# )

# dow =\
# (
#     dow
#     .dropna(axis = 0)
# )

# dow.shape

#### Calculating Linear Daily Return (`NOT` Log Return)

> Remembert this?
<br>

> `Log returns` (a. k. a. continuously compounded returns) are commonly used in quantitative finance for several reasons:

* `Statistical Properties`: Log returns are approximately normally distributed for many assets, especially when considered over short intervals. This normality simplifies various statistical analyses and hypothesis testing.
<br>

* `Time Additivity`: Log returns are additive across time. For instance, if you have daily log returns, you can easily compute weekly, monthly, or yearly log returns by simply summing the daily log returns over those periods. This is not the case with simple (arithmetic) returns, which need to be compounded.
<br>

* `Small Values for Small Changes`: For assets that don't exhibit large price changes over short time horizons, log returns will remain small, making them easier to work with analytically.
<br>

* `Numerical Stability`: Log returns can provide more numerical stability in certain mathematical and computational operations. For instance, when prices of an asset are multiplied by factors (like in stock splits or dividend payments), log returns remain unaffected, whereas simple returns would change.
<br>

* `Economic Interpretations`: In certain economic and financial theories, continuously compounded returns (log returns) have more straightforward interpretations. For example, the Black-Scholes model for option pricing assumes that stock prices follow a geometric Brownian motion, which inherently deals with log returns.
<br>

* `Symmetry`: Log returns are symmetric. This means that a 10% increase in price followed by a 10% decrease in price will result in a net zero log return, but not a net zero arithmetic return.

### `Asset Additivity vs. Time Additivity`

Both `asset additivity` and `time additivity` are crucial in finance. The choice between log returns and simple returns plays an essential role in these considerations.

### Asset Additivity:

- **Definition**: This concept refers to the ability to compute the combined return of several assets as the weighted sum of their individual returns.
  
- **Simple Returns and Asset Additivity**: 
    - Simple returns are asset-additive. 
    - Given two assets in a portfolio, if \( r_1 \) and \( r_2 \) are the simple returns and \( w_1 \) and \( w_2 \) are their respective weights, then the portfolio return \( R \) is:
        $$ R = w_1 r_1 + w_2 r_2 $$

### Time Additivity:

- **Definition**: This concept refers to the ability to compute the combined return over multiple time periods as the sum (or some other function) of the returns from individual periods.

- **Log Returns and Time Additivity**: 
    - Log returns are time-additive.
    - If \( r_{t1} \) is the log return from time \( t_1 \) to \( t_2 \) and \( r_{t2} \) is the log return from time \( t_2 \) to \( t_3 \), the cumulative log return from \( t_1 \) to \( t_3 \) is: 
        $$ r_{t1} + r_{t2} $$
  
- **Simple Returns and Time Additivity**: 
    - Simple returns are not directly time-additive. 
    - To compute the cumulative simple return over multiple periods, one would multiply the gross returns of each period and subtract one.

### Implications:

- For portfolio returns with multiple assets over a single period, the asset additivity of simple returns is more intuitive.
  
- For returns spanning multiple time periods, the time additivity of log returns makes them more suitable.

> In eigenportfolio analysis, capturing the variance-covariance structure of assets over single periods is often the focus. Hence, the asset additivity of simple returns is more directly relevant. However, for time series analysis or compounded returns over longer frames, log returns could be more appropriate.


In [ ]:
# Daily_Linear_Return =\
# (
#     dow
#     .pct_change(1)
# )

# Daily_Linear_Return.head()

In [ ]:
# # Operational defition of outliers = data points beyond 3 SD

# Daily_Linear_Return =\
# (
#     Daily_Linear_Return[Daily_Linear_Return 
#                         .apply(lambda x:(x - x.mean()
#                                         ).abs() < (3 * x.std()
#                                                   )
#                               )
#                         .all(1)
#     ]
# )

In [ ]:
# dow.shape[0] - Daily_Linear_Return.shape[0]

#### Important considerations in data transformation for PCA

* All the variables should be on the same scale before applying PCA; otherwise, a feature with large values will dominate the result. Below, we use StandardScaler in sklearn to standardize the dataset’s features onto unit scale (mean = 0 and variance = 1).
<br>

* Standardization is a useful technique to transform attributes to a standard Normal distribution with a mean of 0 and a standard deviation of 1.

In [ ]:
# %whos

In [ ]:
# scaler =\
# (
#     StandardScaler()
#     .fit(Daily_Linear_Return)
# )

In [ ]:
# scaler

In [ ]:
# scaled_dow =\
# (
#     pd
#     .DataFrame(scaler.fit_transform(Daily_Linear_Return),
#                columns = Daily_Linear_Return.columns,
#                index = Daily_Linear_Return.index)
# )

# scaled_dow.describe()

In [ ]:
# scaled_dow.dtypes

In [ ]:
# plt.figure(figsize = [16, 6]
#           )

# plt.title("AAPL Return")

# plt.ylabel("Linear Return")

# (
#     scaled_dow
#     ["AAPL"]
#     .plot()
# )

### MODEL

#### Data Split

> Let’s divide the portfolio into training and testing data split to execute the analysis regarding the best portfolio and backtesting down the line.

In [ ]:
# prop =\
#     int(len(scaled_dow) * 0.80)

# X_Train = scaled_dow[    : prop] # First 80% of the data
# X_Test  = scaled_dow[prop:     ] # Remaining 20% of the data

# X_Train_Raw = Daily_Linear_Return[    :prop]
# X_Test_Raw  = Daily_Linear_Return[prop:    ]

In [ ]:
# stock_tickers =\
# (
#  scaled_dow
#  .columns
#  .values
# )

# stock_tickers

### Apply Principal Component Analysis

> Let’s run a function to compute principal component analysis using the sklearn module. We run create a function that computes an inversed elbow chart showing the number of principle components and how many of them explain the variance threshold.

### Fitting: Model Comparison with ML Algorithms

In [ ]:
# %whos

In [ ]:
# pca = PCA()

# PrincipalComponent = pca.fit(X_Train)

In [ ]:
# PrincipalComponent

#### First Principal Component /Eigenvector

In [ ]:
# dir(pca)

In [ ]:
# pca.components_[0]

> `pca.components_` is a matrix where each row is a principal component, and the columns correspond to the original features of the data. The components are sorted by their explained variance, with the first component explaining the most variance.

> `pca.components_[0]` retrieves the first row of the matrix, which is the first principal component. It is an eigenvector that points in the direction of the highest variance in the data after accounting for the variance captured by earlier components.

### Explained Variance

> Let’s look at the variance explained using PCA. The decline in the variance of the original data explained by each principal component reflects the correlation among the original features. 

> The eigenvectors with the lowest eigenvalues describe the least amount of variation within the dataset. These values can be dropped. Let's display the number of principal components and the variance explained by each.

In [ ]:
# NumEigenValues = 10

In [ ]:
# fig, axes =\
# (
#     plt
#     .subplots(ncols = 2,
#               figsize = [16, 6]
#              )
# )

# # Plot on the left panel

# Series1 =\
# (
#     pd
#     .Series(pca
#             .explained_variance_ratio_[ :NumEigenValues]
#            )
#     .sort_values()
#     * 100
# )

# # Plot on the right panel

# Series2 =\
# (
#     pd
#     .Series(pca
#             .explained_variance_ratio_[ :NumEigenValues]
#            )
#     .cumsum()
#     * 100
# )

# (
#     Series1
#     .plot
#     .barh(ylim = (0, 9),
#           title = "Explained Variance Ratio by Top 10 PCs",
#           ax = axes[0]
#          )
# )

# (
#     Series2
#     .plot(ylim = (0, 100),
#           xlim = (0, 9),
#           title = "Cumulative Explained Variance by Each PC",
#           ax = axes[1]
#          )
# )

> The first principal component captures the most variance in the original data; the second component represents the second highest variance; and so on.

In [ ]:
# (
#     pd
#     .Series(np
#            .cumsum(pca
#                    .explained_variance_ratio_)
#            )
#     .to_frame("Explained Variance")
#     .head(NumEigenValues)
#     .style
#     .format("{:,.2%}".format)
# )

### Portfolio Weights

> Let’s take a close look at individual principal components. The features may be less interpretable than the original features now. Yet, we can look at the weights of the factors on each principal component to examine any intuitive themes relative to the 28 stocks.

In [ ]:
# pca.components_

In [ ]:
# def PCWeights():

#     weights = pd.DataFrame()

#     for i in range(len(pca.components_)
#                   ):
#         weights["weights_{}".format(i)] = pca.components_[i] / sum(pca.components_[i]
#                                                                   )

#     weights = weights.values.T
#     return weights # Team, be careful with indentation

#### A Step-by-Step Guidance

> A new empty DataFrame, weights, is initialized.

> The function loops through each principal component (eigenvector) in pca.components_.

> For each component, it calculates the normalized weights by dividing each component by the sum of its values. This ensures that the weights for each component sum up to 1.

> These normalized weights are added as a new column to the weights DataFrame.

> After processing all components, the DataFrame is transposed (`.values.T`) to have components as rows and the original features as columns.

* The resulting matrix will have each principal component's normalized weights, sorted by the order of explained variance (i.e., the first row corresponds to the first principal component, the second row to the second component, and so on).

In [ ]:
# weights = PCWeights()

In [ ]:
# weights[0]

> Let’s construct five portfolios, defining the weights of each stock as each of the first five principal components. Then, let’s create a scatterplot that visualizes an organized descending plot with the respective weight of every company at the current chosen principal component.

In [ ]:
# # Set the number of principal components to be considered
# NumComponents = 5

# # Extract the top principal components from the PCA object
# # and create a DataFrame with columns named after the original features

# topPortfolios =\
# (
#     pd
#     .DataFrame(pca.components_[ : NumComponents],
#                columns = dow.columns)
# )

# # Normalize the weights of the top portfolios such that the weights sum up to 1 for each portfolio
# # This is done by dividing each weight by the sum of weights for the respective portfolio

# eigen_portfolios =\
# (
#     topPortfolios
#     .div(topPortfolios.sum(1),
#          axis = 0)
# )

# # Rename the index of the eigen_portfolios DataFrame for better readability

# eigen_portfolios.index = [f"Portfolio {i}" for i in range(NumComponents)
#                          ]

# # Calculate the square root of the explained variance for each component
# # This provides the standard deviation of returns for each eigenportfolio

# np.sqrt(pca.explained_variance_)

In [ ]:
# eigen_portfolios

In [ ]:
# eigen_portfolios.iloc[0]

In [ ]:
# (
#     eigen_portfolios
#     .T  # Transpose the DataFrame to have portfolios as columns and assets as rows
#     .plot
#     .bar(subplots = True,
#          layout = (int(NumComponents), 1),
#          legend = False,
#          sharey = True,
#          figsize = [16, 20],
#          ylim = [-1, 1]
#         )
# )

> The heatmap and the plot above shown the contribution of different stocks in each eigenvector.

In [ ]:
# plt.figure(figsize = [16, 6]
#           )

# sns.heatmap(topPortfolios,
#             cmap = "viridis")

### Equal-weigthed Portfolio vs. PCA-based Portfolio

> An equal-weighted portfolio is a portfolio in which all assets are given the same weight, regardless of their individual risk and return characteristics. This means that each asset contributes equally to the overall portfolio, regardless of its performance.

> In contrast, a PCA-based portfolio (an eigen portfolio) is a portfolio that is constructed using a mathematical technique called principal component analysis (PCA). PCA is a statistical technique used to identify the underlying factors that explain the co-movement of asset returns. The eigen portfolio is a portfolio that is constructed using the eigenvectors of the covariance matrix of the asset returns.

> An eigen portfolio may be better than an equal-weighted portfolio for several reasons:

- `Improved Diversification`: The eigen portfolio is designed to capture the most important sources of risk in the market. By investing in these sources of risk, the eigen portfolio may provide improved diversification compared to an equal-weighted portfolio, which may be more exposed to idiosyncratic risk.
<br>

- `Improved Risk-Adjusted Returns`: The eigen portfolio is designed to capture the factors that explain the most variance in the market. By investing in these factors, the eigen portfolio may provide improved risk-adjusted returns compared to an equal-weighted portfolio, which may be more exposed to less important factors.
<br>

- `More Efficient`: The eigen portfolio is constructed using a mathematical technique that is designed to be more efficient than an equal-weighted portfolio. This means that the eigen portfolio may provide better risk-adjusted returns for a given level of risk.
<br>

Taken together, an eigen portfolio may provide better risk-adjusted returns and improved diversification compared to an equal-weighted portfolio. However, it's important to note that the performance of any portfolio will depend on the specific assets included and the market conditions.

### How to Find the Best Eigen Portfolio

> To determine the best eigen portfolio, let’s employee the Sharpe ratio. As you have learned, the Sharpe ratio is an assessment of risk-adjusted performance that explains the annualized returns against the annualized volatility of a portfolio.

In [ ]:
# def calculate_sharpe_ratio(ts_returns, periods_per_year = 252):

#     n_years = ts_returns.shape[0] / periods_per_year

#     annualized_return = np.power(np.prod(1 + ts_returns), (1 / n_years)
#                                 ) - 1

#     annualized_vol = ts_returns.std() * np.sqrt(periods_per_year)

#     annualized_sharpe = annualized_return / annualized_vol

#     return annualized_return, annualized_vol, annualized_sharpe

> Let's construct a loop to compute the principle component’s weights for each eigen portfolio, which then uses the sharpe ratio function to look for the portfolio with the highest sharpe ratio. 

### Backtesting Our `Eigen` Portfolio

> Let’s backtest our algorithm on the test set. We will look at a few of the top performers and the worst performer.
 
> The outperformance or underperformance here is attributed to the weights of the stocks or sectors in the eigen portfolio. We can examine these further to identify the individual drivers of each portfolio.

In [ ]:
# def valid_backtest_PCA_porfolios(eigen):
    
#     eigen_prtfi =\
#         (
#             pd
#             .DataFrame(data = {"weights": eigen.squeeze()
#                               },
#                        index = stock_tickers)
#         )

#     # Sanity Check: Ensure the order of tickers in X_Test_Raw matches the order in eigen
#     if not list(eigen_prtfi.index) == list(X_Test_Raw.columns): 
#         raise ValueError("Sanity check failed: Mismatch in number of tickers between X_Test_Raw and eigen.")
#     else:
#         print("Prof. Roh's Message: 'Sanity check succeeded :)' The order of tickers in X_Test_Raw matches the order in eigen.")
    
#     # Let's directly compute the dot product without sorting
#     eigen_prtfi_returns =\
#     (
#         np
#         .dot(X_Test_Raw, eigen)
#     )
    
#     eigen_portfolio_returns =\
#     (
#         pd
#         .Series(eigen_prtfi_returns.squeeze(),
#                 index = X_Test_Raw.index)
#     )

#     returns, vol, sharpe = calculate_sharpe_ratio(eigen_portfolio_returns)

#     print("Our PCA-based Portfolio:\nReturn = %.2f%%\nVolatility = %.2f%%\nSharpe = %.2f"  %
#           (returns * 100, vol * 100, sharpe)
#          )

#     # Compared with what? Equal-weightage Portfolio

#     equal_weight_return =\
#     (
#         X_Test_Raw * (1 / len(pca.components_)
#                      )
#     ).sum(axis = 1)

#     df_plot =\
#         (
#             pd
#             .DataFrame({"ML Portfolio Return": eigen_portfolio_returns,
#                         "Equal Weight Index": equal_weight_return},
#                       index = X_Test.index
#                       )
#         )

#     (
#         np
#         .cumprod(df_plot + 1)
#         .plot(title = "Returns of the equal weighted index vs. Eigen-Portfolio",
#               figsize = [16, 8]
#              )
#     )

#     plt.show()

In [ ]:
# valid_backtest_PCA_porfolios(eigen = weights[5]
#                              )

    NOTE
    
> Given that these eigen portfolios are independent, they also provide diversification opportunities. As such, we can invest across these uncorrelated eigen portfolios, providing other potential portfolio management benefits.

> `Thank you for working with the script, Team 👍`